In [ ]:
import sys
import os
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
import torch
import copy
from functools import partial
from topic_watermark_processor import TopicWatermarkLogitsProcessor
from semantic_topic_extension import EmbeddingMapper
from model import (
    load_model, 
    load_sentence_model, 
    load_topic_model,
    detect,
)
from transformers import (
    LogitsProcessorList,
) 
from datasets import load_dataset
import numpy as np
import json


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = {
    # 'model_name_or_path': 'facebook/opt-2.7b',
#     'model_name_or_path': 'facebook/opt-1.3b',
#     'model_name_or_path': 'facebook/opt-125m',
    'model_name_or_path': 'facebook/opt-6.7b',
#     'model_name_or_path': 'google/gemma-7b',
    'load_fp16' : False,
    'prompt_max_length': None, 
    'max_new_tokens': 200, 
    'generation_seed': 123, 
    'use_sampling': True, 
    'n_beams': 1, 
    'sampling_temp': 0.7, 
    'use_gpu': True, 
    'seeding_scheme': 'simple_1', 
    'delta': 10.0, 
    'ignore_repeated_bigrams': False, 
    'detection_z_threshold': 4.0, 
    'select_green_tokens': True,
    'skip_model_load': False,
    'seed_separately': True,
    'topic_token_mapping': {},
    'granularity': 'kmeans',
}

In [ ]:
def compute_distinct_n(text, tokenizer, n=1):
    tokens = tokenizer.tokenize(text)
    if len(tokens) < n:
        return 0.0
    
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    distinct_ngrams = len(set(ngrams))
    distinct_score = distinct_ngrams / len(ngrams)
    
    return distinct_score

In [ ]:
topic_list = ["animals", "technology", "sports", "medicine"]

model, tokenizer = load_model(args)
print("Model loaded")

In [ ]:
c4_dataset = load_dataset('json', data_files='./c4/c4.json', split='train[:1000]')
c4_iter = iter(c4_dataset)
n = 100  # how many prompts to sample
inputs_list = []
for _ in range(n):
    entry = next(c4_iter)
    text = entry["text"]
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    prompt_ids = token_ids[:100]
    prompt = tokenizer.decode(prompt_ids, clean_up_tokenization_spaces=True)
    inputs_list.append(prompt)

In [ ]:
sentence_model = load_sentence_model()
print("Sentence model loaded")
embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)
total_tokens, vocab_embeddings = embedding_mapper.get_model_vocab_embeddings()
topic_embeddings = embedding_mapper.get_defined_topic_list_embeddings(topic_list)
topic_token_mapping = embedding_mapper.map_tokens_to_topics(total_tokens, vocab_embeddings, topic_list, topic_embeddings, threshold=0.7)
args['topic_token_mapping'] = topic_token_mapping

In [ ]:
def generate_TBW(prompt, detected_topics, args, model=None, tokenizer=None, sentence_model=None):
  
    detected_topic = ''
    for topic in detected_topics:
        if topic.lower() in args['topic_token_mapping']:
            detected_topic = topic.lower()
            print(f"Topic detected in one to one mapping: {detected_topic}")
            break
    if detected_topic == '':
        embedding_mapper = EmbeddingMapper(tokenizer, sentence_model)

        if args['granularity'] == 'kmeans':
            print(f"Mapping topic with granularity Kmeans.")
            detected_topic, _ = embedding_mapper.kmeans_detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics,
            )
        else:
            print(f"Mapping topic with granularity averaging")
            detected_topic, _ = embedding_mapper.detected_topics_to_embeddings(
                list(args['topic_token_mapping'].keys()), 
                detected_topics
            )

    print(f"Detected topic: {detected_topic}")
    watermark_processor = TopicWatermarkLogitsProcessor(
        vocab=list(tokenizer.get_vocab().values()),
        delta=args['delta'],
        seeding_scheme=args['seeding_scheme'],
        select_green_tokens=args['select_green_tokens'],
        topic_token_mapping=args['topic_token_mapping'],
        detected_topic=detected_topic,
    )
  
    gen_kwargs = {
        'max_new_tokens': args['max_new_tokens'],
        'min_new_tokens': args['min_new_tokens']
    }
    
    if args['use_sampling']:
        gen_kwargs.update(dict(
            do_sample=True, 
            top_k=0,
            temperature=args['sampling_temp'],
#             repetition_penalty=1.1,
        ))
    else:
        gen_kwargs.update(dict(
            num_beams=args['n_beams']
        ))

    generate_with_watermark = partial(
        model.generate,
        logits_processor=LogitsProcessorList([watermark_processor]), 
        **gen_kwargs
    )

    if args['prompt_max_length']:
        pass
    elif hasattr(model.config,"max_position_embeddings"):
        args['prompt_max_length'] = model.config.max_position_embeddings - args['max_new_tokens']
    else:
        args['prompt_max_length'] = 2048 - args['max_new_tokens']

    tokenized_input = tokenizer(
        prompt, 
        return_tensors="pt", 
        add_special_tokens=True, 
        truncation=True, 
        max_length=args['prompt_max_length']
    ).to(device)

    output_with_watermark = generate_with_watermark(**tokenized_input)

    if args['decoder']:
        # need to isolate the newly generated tokens
        output_with_watermark = output_with_watermark[:,tokenized_input["input_ids"].shape[-1]:]

    decoded_output_with_watermark = tokenizer.batch_decode(output_with_watermark, skip_special_tokens=True)[0]

    return decoded_output_with_watermark

In [ ]:
warmup_length = 10
schemes = {
    "TBW": generate_TBW,
}

input_text = inputs_list[0]
detected_topics = load_topic_model(input_text)
args_warmup = copy.deepcopy(args)
args_warmup['max_new_tokens'] = warmup_length
args_warmup['min_new_tokens'] = warmup_length
generated_text = generate_TBW(input_text, detected_topics, args_warmup, model=model, tokenizer=tokenizer, sentence_model=sentence_model)
print("Warmup done")


quality_tokens = args['max_new_tokens']
results = {scheme: [] for scheme in schemes}
generated_text_per_scheme = {}
for scheme, func in schemes.items():
    qualities = []
    texts_w = []
    
    for input_text in inputs_list:
        detected_topics = load_topic_model(input_text)
        
        args_iter = copy.deepcopy(args)
        args_iter['max_new_tokens'] = quality_tokens+5
        args_iter['min_new_tokens'] = quality_tokens-5
        generated_text = func(input_text, detected_topics, args_iter, model=model, tokenizer=tokenizer, sentence_model=sentence_model)
        texts_w.append(generated_text)
        distinct_1 = compute_distinct_n(generated_text, tokenizer, n=1)
        distinct_2 = compute_distinct_n(generated_text, tokenizer, n=2)
        distinct_3 = compute_distinct_n(generated_text, tokenizer, n=3)
        distinct_4 = compute_distinct_n(generated_text, tokenizer, n=4)
        distincts = {'distinct_1': distinct_1, 'distinct_2': distinct_2, 'distinct_3': distinct_3, 'distinct_4': distinct_4}
        qualities.append(distincts)
        print(f"{scheme}: {distincts}")
    
    results[scheme] = qualities
    generated_text_per_scheme[scheme] = texts_w
    

In [ ]:
schemes_detection = {
    "TBW": detect,
}

detected_results = {}
for scheme, texts in generated_text_per_scheme.items():
    print(scheme)
    detection_function = schemes_detection.get(scheme)
    detection_results = []
    for idx, text in enumerate(texts):
        args_iter = copy.deepcopy(args)
        args_iter['max_new_tokens'] = quality_tokens+5
        args_iter['min_new_tokens'] = quality_tokens-5
        result_formatted = detection_function(inputs_list[idx], text, args_iter, device=device, tokenizer=tokenizer, sentence_model=sentence_model)
        result = {
            'is_watermarked': result_formatted[6][1] == 'Watermarked',
            'score': float(result_formatted[3][1])
        }
        print(result)
        detection_results.append(result)
    detected_results[scheme] = detection_results

In [ ]:
combined_results = {}
for scheme, distinct_list in results.items():
    combined_results[scheme] = {}
    
    for metric in ['distinct_1', 'distinct_2', 'distinct_3', 'distinct_4']:
        values = [entry[metric] for entry in distinct_list]
        mean_val = np.mean(values)
        std_val = np.std(values)
        combined_results[scheme][metric] = {"mean": mean_val, "std": std_val}
    
    score_list = detected_results.get(scheme, [])
    scores = [entry['score'] for entry in score_list]
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    combined_results[scheme]["score"] = {"mean": mean_score, "std": std_score}

for scheme, metrics in combined_results.items():
    print(f"Scheme: {scheme}")
    for metric, stats in metrics.items():
        print(f"  {metric}: mean = {stats['mean']:.2f}, std = {stats['std']:.2f}")
    print()